# Automation of the <span style="color: #04A9AF">foryouadyourcustomers</span> icon library

The icon library source assets are located on BOX: https://foryouandyourcustomers.box.com/s/5fn5tl15qo741kyo8du0

Cloudinary (CDN): https://cloudinary.com/console/c-8efc42db506d8fb4a8ada3f7e707b2/media_library/folders/home

Credentials: To be defined


Workflow:
1. Collect files form BOX
1. Load content from moderated confluence page
1. Print summary
1. Convert pdf to svg
1. Upload to cloudinary
1. Produce generated page content


## Configuration

In [ ]:
# asset will be publicliy available
access = 'upload'
# assset will be private (key in url)
#access = 'private'

target_folder = 'icon_library'

output_folder = 'output'

In [ ]:
# File type constants
PDF = 'pdf'
PNG = 'png'
SVG = 'svg'

In [ ]:
import os
import logging
log = logging.getLogger(__name__)

In [ ]:
os.makedirs(output_folder, exist_ok=True)

## Collect files from BOX

In [ ]:
from pathlib import Path

box_library_root = os.environ['HOME'] + '/Box/Q-bis_Ende_2019/3/31/MediaAssets/Official_Icon_Library'
assert os.path.isdir(box_library_root), "Is directory"

png_list = list(Path(box_library_root).rglob("*.[pP][nN][gG]"))
pdf_list = list(Path(box_library_root).rglob("*.[pP][dD][fF]"))
log.warning('Found {} PDF and {} PNG images'.format(len(png_list), len(pdf_list)))
all_assets = pdf_list + png_list
all_assets[:3]

In [ ]:
def fix_id(identifier):
    ''' Icon identifier is a 4 digit number with leading zeros '''
    i = int(identifier)
    return '{0:04d}'.format(i)

assert fix_id('01') == '0001'
assert fix_id('502') == '0502'

In [ ]:
import re
filename_id_extractor = re.compile(".*_([0-9]{3,4})_(.+)\.(png|pdf)", re.IGNORECASE)

def icon_id(file):
    ''' Extracts the icon id from properly formatted filenames '''
    base = os.path.basename(file)
    match = filename_id_extractor.match(base)
    if match:
        return fix_id(match.group(1))
    else:
        log.warning('Unable to extract icon id from path ' + str(file))
        return None

assert '0' == icon_id(all_assets[0])[0], 'Id should start with a leading 0'

In [ ]:
assets = []

def range_generator():
    n = 9000
    while True:
        yield n
        n += 1

generator = range_generator()
        
for file in all_assets:
    filename = os.path.basename(file)
    fragments = os.path.splitext(filename)
    file_name = fragments[0]
    file_extension = fragments[1]
    parent_folder, _ = os.path.split(file)
    _, folder_name = os.path.split(parent_folder)
    file_type_folder = PNG if re.match('\.png', file_extension, re.IGNORECASE) else PDF
    key = icon_id(file)
    if not key:
        key = fix_id(next(generator))
    asset = { 
        'id': key,
        'filename': file_name, 
        'extension': file_extension, 
        'folder': folder_name, 
        'type': file_type_folder, 
        'source': str(os.path.abspath(file)) }
    assets.append(asset)
    
assets[:2]

## Combine data sources

In [ ]:
# map with key icon id (if missing, negative numbers are automatically created)
icon_map = {}

unassigned_range = 9000

for file in assets:
    file_id = file['id']
    if not file_id:
        file_id = unassigned_range
        unassigned_range += 1
        log.warning('File {} carries no id'.format(file['source']))
    key = fix_id(file_id)
    entry = icon_map.get(key, {})
    entry[file['type']] = file
    icon_map[key] = entry
    entry['filename'] = file['filename']
    entry['folder'] = file['folder']

In [ ]:
print('{} files have nonconforming filename of {} files'.format(unassigned_range - 9000, len(assets)))

singles = list(filter(lambda identifier : len(identifier) < 2, icon_map.values()))
print('Singel media format assets: {}'.format(len(singles)))

In [ ]:
icon_map['0004']

# Read metadata from Confluence

Page https://foryouandyourteam.com/x/y4f2Aw contains the moderated metadata.

This page's source is exported to the file `icon_library_confluence_content.html` int the BOX root folder of the icon library for simpler parsing.

In [ ]:
from lxml import etree

tree = None
with open(box_library_root + '/icon_library_confluence_content.html', 'r') as f:
    content = f.read()
    tree = etree.HTML('<html><body>' + content + '</body></html>')

tables = tree.findall('.//table//tbody')

# there are currently 6 tables on the confluence page ...
assert len(tables) > 1

'{} tables found in Confluence content'.format(len(tables))

In [ ]:
rows = []

def extract_cell_content(cell):
    return cell.text.strip()

def content(entity):
    return ''.join(entity.xpath('.//child::text()')).strip()
        
def fix_columns(columns: list):
    if len(columns) > 8:
        '''First table currenty contains bogus column 1'''
        #print('{} {} exceedes 8 columns'.format(columns[0], columns[3]))
        del columns[1] 

insensitive_png = re.compile(re.escape('png'), re.IGNORECASE)

def row_to_record(element, cells, topic, missing_id_generator):
    cell0_text = cells[0].text
    if cell0_text and cell0_text.strip().isnumeric():
        number = int(cell0_text)
#        print('found {}: {}'.format(cell0_text, cells[3].text))
        rows.append({ 'number': number, 'cells': cells, 'text': cells[3].text})
        ct = list(map(lambda x: content(x), list(cells)))
#        print('content: {}'.format(list(ct)))
        fix_columns(ct)
        
        identity = fix_id(ct[0])
        
        name_en = ct[2]
        name_de = ct[3]
        
        keywords_en = ct[4]
        keywords_de = ct[5]
        
        has_png = 'missing'
        has_pdf = 'missing'
        icon = icon_map.get(identity)
        filename = None
        if icon:
            png = icon.get(PNG)
            if png:
                has_png = 'PNG'
                filename = png['filename']
            pdf = icon.get(PDF)
            if pdf:
                has_pdf = 'PDF'
                filename = pdf['filename']
        else:
            icon = {}
            identity = fix_id(next(missing_id_generator))
            icon['id'] = identity
            icon_map[identity] = icon
            log.warning('New id {} for {} since no BOX asset has been found'.format(identity, ct))
            
        # try to fix the filename
        if not filename:
            img = element.find('.//attachment')
            if img:
                print(img.attrib['key'])
                #filename = img['ri:value']
            filename = '???'
            
        # Add data to icon map
        icon['keywords'] = { 'de': keywords_de, 'en': keywords_en }
        icon['description'] = { 'de': name_de, 'en': name_en }
        icon['filename'] = filename
        
        ct.append(filename)
        icon['folder'] = icon.get('folder', topic)
        

# This should not be hardcoded here
topics_as_on_page_order = [ 'ChannelOverview', 
                            'Additional Channels', 
                            'Objects', 
                            'Tangible Objects', 
                            'Places and Buildings',
                            'Persons',
                            'Services',
    ] 

topic_sequence = (topic for topic in topics_as_on_page_order)

for table in tables:
    topic = next(topic_sequence)
    # collect all rows
    context = etree.iterwalk(table, events=("start","end"), tag="tr")
    for action, element in context:
        cells = element.findall('.//td')
        if len(cells) > 3 and "start" in action:
            row_to_record(element, cells, topic, generator)



In [ ]:
preview_columns =['ID', 'Preview', 'Name (de)', 'Schlüsselwörter', 'Name (en)', 'Keywords', 'Filename', 'Folder', 'PNG', 'PDF']

records = []

for key in icon_map:
    # see preview_columns
    icon = icon_map[key]
    description = icon.get('description')
    if description:
        text_de = description.get('de')
        text_en = description.get('en')
    else:
        text_de = ''
        text_en = ''
    keywords = icon.get('keywords')
    if keywords:
        keywords_de = keywords.get('de')
        keywords_en = keywords.get('en')
    else:
        keywords_de = ''
        keywords_en = ''
    records.append( [ key, int(key), text_de, keywords_de, text_en, keywords_en, icon['filename'], icon['folder'], 
            'PNG' if icon.get(PNG) else '', 'PDF' if icon.get(PDF) else '' ] )

records[:2]

In [ ]:
len(records)

In [ ]:
records = sorted(records, key = lambda r : r[1])
records[2:]

# Merge confluence and asset lists
Combine asset files with confluence metadata

In [ ]:
icon_map['0067']

# Export

## Write ssot jaml

In [ ]:
yaml_output = { 'version': '0.1', 'date': '2021-02-03 8:20',
    'icons': icon_map
}

In [ ]:
import yaml

output_folder = 'output'
os.makedirs(output_folder, exist_ok=True)
    
with open(os.path.join(output_folder, 'ssot.yaml'), 'w') as outfile:
     yaml.dump(yaml_output, outfile)

## Information table prview

In [ ]:
import pandas as pd

# what is number used for?
df = pd.DataFrame(records, columns = preview_columns)
df

In [ ]:
records[2:]

## PDF files will be converted to SVG on the fly
Uses pdf2svg tool

In [ ]:
import tempfile
svg_output_folder = os.path.join(output_folder, 'svg')
os.makedirs(svg_output_folder, exist_ok=True)
svg_output_folder

In [ ]:
import os

import subprocess
result = subprocess.run("pdf2svg --help", shell=True, check=False, capture_output=True)
assert result.stdout.decode().find('Usage') >= 0, 'Missing pdf2svg! Install using `brew install pdf2svg`'

def create_svg(file, svg_output_folder):
    if file.get(SVG) and os.path.isfile(file[SVG]):  # Shortcut if SVG is already there
        return file[SVG]
    abs_path_source = os.path.abspath(file['source'])
    assert os.path.isfile(abs_path_source), 'Source file {} does not exist'.format(abs_path_source)
    abs_path_destination = os.path.join(svg_output_folder, '{}-{}.svg'.format(file['id'], file['filename']))
    convert_result = subprocess.run(['pdf2svg', abs_path_source, str(abs_path_destination)], check=False, capture_output=True)
    if convert_result.returncode != 0:
        log.warning('Unable to convert {} to svg'.format(abs_path_source))
    file[SVG] = abs_path_destination
    return abs_path_destination

create_svg({ 'id': 'test', 'filename': 'f-icon_person_0270_goup8', 'source': 'testdata/f-icon_person_0270_goup8.pdf'}, svg_output_folder)

In [ ]:
import copy
import ipywidgets
from IPython.display import display  

pdf_assets = list(filter(lambda icon: icon.get(PDF), icon_map.values()))
svg_asserts = []

def generate_svg(asset):
    #print('Converting asset {}'.format(asset))
    local_svg_image = create_svg(asset[PDF], svg_output_folder)
    svg = copy.deepcopy(asset)[PDF]
    #print(svg)
    svg['extension'] = '.svg'
    svg['source'] = local_svg_image
    svg['type'] = SVG
    asset[SVG] = svg
    svg_asserts.append(svg)
    

progress_bar = ipywidgets.IntProgress(min=0, max=len(pdf_assets), 
    description='Converting PDF to SVG ({})'.format(len(pdf_assets)),
    layout={'width': '100%'}, style = {'description_width':'initial'})
display(progress_bar)

for asset in pdf_assets:
    progress_bar.value += 1
    generate_svg(asset)

progress_bar.bar_style = "success"

## Visual preview

In [ ]:
def imgid_to_icon(identifier):
    icon = icon_map.get(fix_id(identifier))
    if icon and icon.get(SVG):
        return '<img src="{}" width="60"/>'.format(icon[PDF]['svg'])
    else:
        return '<b>Missing icon {}</br>'.format(identifier)

tablecode = df.to_html(escape=False, formatters=dict(Preview=imgid_to_icon))

In [ ]:
from IPython.display import display
from IPython.core.display import HTML
display(HTML(tablecode))

In [ ]:
with open(os.path.join(output_folder, 'icon_library.html'), 'w') as output:
           output.write(tablecode.replace('src="output/svg/', 'src="svg/'))

## Cloudinary upload

According to: https://cloudinary.com/documentation/upload_images#uploading_with_a_direct_call_to_the_rest_api

URL schema: `https://api.cloudinary.com/v1_1/<cloud name>/<resource_type>/upload`

Replace the trailing **upload** with **private** to restrict access using an encoded key in the url.

In [ ]:
import hashlib

def signature(params, secret):
    ''' 
        Calculates the sha signature according to 
        https://cloudinary.com/documentation/upload_images#generating_authentication_signatures
    '''
    sorter = list()
    for entry in params:
        sorter.append(entry + '=' + params[entry] + '&')
    sorter.sort()
    serialized = ''.join(sorter)
    serialized =serialized[:-1] + secret
    #print(serialized)
    m = hashlib.sha256()
    m.update(serialized.encode('utf-8'))
    return m.hexdigest()

# eager=w_400,h_300,c_pad|w_260,h_200,c_crop&public_id=sample_image
sample = { 'timestamp': '1315060510', 'eager': 'w_400,h_300,c_pad|w_260,h_200,c_crop', 'public_id': 'sample_image' }
sig = signature(sample, 'abcd') 
sig

In [ ]:
def metadata(icon, file, param: dict):
    
    tags = ','.join([ icon['folder'], 'icon_library', 'fyayc' ])

    metafields =  [ 'id=' + file['id'] ]
    description = icon.get('description')
    if description:
        metafields.append( 'title=' + description.get('en', '') )
        metafields.append( 'title_de=' + description.get('de', '') )
    
    keywords = icon.get('keywords')
    if keywords:
        metafields.append( 'keywords=' + keywords.get('en', '') )
        metafields.append( 'keywords_de=' + keywords.get('de', '') )
      
        
    
    #meta = 'caption={}|alt={}|titel={}|keywords={}|schluesselwoerter={}'.format(
      
    param['tags'] = tags
    # we use context instead of controlled 'metadata'
    param['context'] = '|'.join(metafields)
    return param

In [ ]:
sample = icon_map['0067']
sample_png = sample[PNG]
metadata(sample, sample_png, {})

In [ ]:
import requests
import base64
import time

time_stamp = int(time.time())

account='fyayc-test' 
uri='https://api.cloudinary.com/v1_1/' + account + '/image/upload'
api_key='778862129561347'
api_secret='-xJwqK-LkCPVdKZrDosLpesBM5Q'

def upload(prefix, icon, file):
    if file.get('url'): # Already uploaded?
        return None
    
    file_data = None
    with open(file['source'], 'rb') as binary_file:
        binary_file_data = binary_file.read()
    
    path = '{}/{}/{}'.format(prefix, file['type'], icon['folder'])
    icon['cloudinary'] = path
    
    params = {}
    params['timestamp'] = str(time_stamp)
    
    # optional parameters
    metadata(icon, file, params)

    params['public_id'] = icon['filename']
    params['folder'] = path
    params['tags'] = 'icon, foryouandyourcustomers, fyayc'
    params['type'] = access
    
    # calculate signature over parameters
    sig = signature(params, api_secret)

    # append non signed parameters
    params['api_key'] = api_key
    params['signature'] = sig

    payload = params
 
    files_data = {'file': (file_name, binary_file_data)}
    resp = requests.post(uri + '?timestamp=' + str(time_stamp) + '&api_key=' + api_key + '&signature=' + sig + '&',
                        data = payload,
                        files = files_data)
    #print(vars(resp))
    response = resp.json()
    if response.get('secure_url'):
        file['url'] = response['secure_url']
        file['cloudinary_upload'] = response
        print('Icon {}{} is now available at {}'.format(file['filename'], file['extension'], file['url']))
    else:
        log.error('Cannot publish icon {}. Params: {}'.format(file, params))
        print(vars(resp))
    return response

sample = icon_map['0067']
sample_png = sample[PNG]
try:
    sample_png.pop('url')
except:
    pass
    
upload('test', sample, sample_png)

In [ ]:
upload_progress = ipywidgets.IntProgress(min=0, max=len(icon_map), 
description='Publishing to Cloudinary ({})'.format(len(icon_map)),
layout={'width': '100%'}, style = {'description_width':'initial'})
display(upload_progress)

prefix = 'fyayc_icon_library_bosch'

for key in icon_map:
    upload_progress.value += 1
    icon = icon_map[key]
    png = icon.get(PNG)
    if png:
        upload(prefix, icon, png)
    svg = icon.get(SVG)
    if svg:
        upload(prefix, icon, svg)
    pdf = icon.get(PDF)
    if pdf:
        upload(prefix, icon, pdf)

upload_progress.bar_style = "success"

# Persist icon map including Cloudinary resource urls

In [ ]:
import yaml
    
with open(os.path.join(output_folder, 'ssot-cloudinary.yaml'), 'w') as outfile:
     yaml.dump(yaml_output, outfile)

# Transfer tags and metadata from confluence to Cloudinary

API documentation: https://cloudinary.com/documentation/metadata_api

Confluence metadate is extracted from confluence page html ...

In [ ]:
import base64

base_url='https://api.cloudinary.com/v1_1/' + account

def list_metadata_fields():
    secret = (api_key + ':' + api_secret).encode('utf-8')
    secret_base64 = base64.b64encode(secret)     
    authstring = secret_base64.decode('utf-8')
    headers = { 'Authorization': 'Basic {}'.format(authstring) }
    resp = requests.get(metadata_url + '/metadata_fields', headers=headers )
    return resp
    
def update_metadata_fields(icon):
    yield
    
    
vars(update_meta_information(icon_map['0067']))

In [ ]:
for key in icon_map:
    icon = icon_map[key]
    